In [1]:

# ============================================================
# ZEROQ AI - NOTEBOOK 02
# DATA CLEANING AND STANDARDIZATION
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT REQUIRED LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

print("Libraries imported successfully.")


# ------------------------------------------------------------
# 2. DEFINE DATASET FILENAMES
# ------------------------------------------------------------

DATASET_FILES = {
    "products": "products.csv",
    "customers": "customers.csv",
    "facilities": "facilities.csv",
    "production_lines": "production_lines.csv",
    "materials": "materials.csv",
    "production_batches": "production_batches.csv",
    "complaints": "complaints.csv",
    "quality_inspections": "quality_inspections.csv",
    "investigations": "investigations.csv",
    "corrective_actions": "corrective_actions.csv",
    "data_dictionary": "data_dictionary.csv"
}

REQUIRED_FILES = set(DATASET_FILES.values())


# ------------------------------------------------------------
# 3. AUTOMATICALLY LOCATE RAW DATA FOLDER
# ------------------------------------------------------------

print("\nSearching for the raw data folder...")

search_roots = [Path.cwd(), *Path.cwd().parents]

BASE_PATH = None

for root in search_roots:

    for file_path in root.rglob("products.csv"):

        candidate_folder = file_path.parent

        available_files = {
            file.name for file in candidate_folder.glob("*.csv")
        }

        if REQUIRED_FILES.issubset(available_files):
            BASE_PATH = candidate_folder
            break

    if BASE_PATH is not None:
        break

if BASE_PATH is None:
    raise FileNotFoundError(
        "Could not locate the folder containing all 11 CSV files. "
        "Open the correct VS Code project folder and rerun."
    )

PROJECT_PATH = BASE_PATH.parent

PREPROCESSING_PATH = PROJECT_PATH / "preprocessing"
PROCESSED_DATA_PATH = PROJECT_PATH / "processed_data"
VISUALIZATION_PATH = PROJECT_PATH / "visualizations"

REPORT_PATH = PREPROCESSING_PATH / "reports"

for folder in [
    PREPROCESSING_PATH,
    PROCESSED_DATA_PATH,
    VISUALIZATION_PATH,
    REPORT_PATH
]:
    folder.mkdir(parents=True, exist_ok=True)

print("\nPROJECT PATHS")
print("-" * 60)
print("Project:", PROJECT_PATH)
print("Raw data:", BASE_PATH)
print("Processed data:", PROCESSED_DATA_PATH)
print("Reports:", REPORT_PATH)


# ------------------------------------------------------------
# 4. LOAD ALL RAW DATASETS
# ------------------------------------------------------------

raw_datasets = {}
cleaned_datasets = {}

for dataset_name, filename in DATASET_FILES.items():

    file_path = BASE_PATH / filename

    try:
        df = pd.read_csv(file_path)

    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding="latin1")

    raw_datasets[dataset_name] = df.copy()

    print(
        f"{dataset_name:25} | "
        f"Loaded successfully | "
        f"Shape: {df.shape}"
    )

print("\nTotal datasets loaded:", len(raw_datasets))

if len(raw_datasets) != 11:
    raise ValueError("Not all 11 datasets were loaded.")


# ------------------------------------------------------------
# 5. DEFINE PRIMARY KEYS
# ------------------------------------------------------------

PRIMARY_KEYS = {
    "products": "product_id",
    "customers": "customer_id",
    "facilities": "facility_id",
    "production_lines": "production_line_id",
    "materials": "material_id",
    "production_batches": "batch_id",
    "complaints": "complaint_id",
    "quality_inspections": "inspection_id",
    "investigations": "investigation_id",
    "corrective_actions": "action_id"
}


# ------------------------------------------------------------
# 6. INITIALIZE CLEANING REPORTS
# ------------------------------------------------------------

cleaning_summary = []
missing_value_reports = []
duplicate_reports = []
primary_key_reports = []
date_reports = []
column_reports = []
numeric_string_reports = []


# ------------------------------------------------------------
# 7. CLEAN EACH DATASET
# ------------------------------------------------------------

for dataset_name, original_df in raw_datasets.items():

    print("\n" + "=" * 75)
    print("CLEANING DATASET:", dataset_name.upper())
    print("=" * 75)

    df = original_df.copy()

    original_rows = len(df)
    original_columns = len(df.columns)

    # --------------------------------------------------------
    # 7.1 STANDARDIZE COLUMN NAMES
    # --------------------------------------------------------

    original_column_names = df.columns.tolist()

    df.columns = [
        str(column).strip()
        for column in df.columns
    ]

    if df.columns.duplicated().any():
        raise ValueError(
            f"Duplicate column names found after trimming in "
            f"{dataset_name}. Inspect the original CSV."
        )

    changed_column_count = sum(
        old != new
        for old, new in zip(original_column_names, df.columns)
    )

    print("Column names standardized:", changed_column_count)

    # --------------------------------------------------------
    # 7.2 STANDARDIZE TEXT VALUES
    # --------------------------------------------------------

    text_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:

        df[column] = df[column].map(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )

        # Convert empty or whitespace-only strings into NaN
        df[column] = df[column].replace(
            r"^\s*$",
            np.nan,
            regex=True
        )

    print("Text columns standardized:", len(text_columns))

    # --------------------------------------------------------
    # 7.3 IDENTIFY AND PARSE DATE COLUMNS
    # --------------------------------------------------------

    date_keywords = [
        "date",
        "timestamp",
        "datetime",
        "created_at",
        "updated_at",
        "closed_at",
        "reported_at",
        "resolved_at"
    ]

    date_columns = [
        column
        for column in df.columns
        if any(
            keyword in column.lower()
            for keyword in date_keywords
        )
    ]

    for column in date_columns:

        original_values = df[column].copy()

        parsed_dates = pd.to_datetime(
            original_values,
            errors="coerce"
        )

        valid_date_count = int(parsed_dates.notna().sum())

        invalid_date_count = int(
            (
                original_values.notna()
                & parsed_dates.isna()
            ).sum()
        )

        # Convert only if at least one value was parsed.
        # If none can be parsed, preserve the original values.
        if valid_date_count > 0:
            df[column] = parsed_dates

        date_reports.append({
            "Dataset": dataset_name,
            "Date_Column": column,
            "Valid_Dates": valid_date_count,
            "Invalid_Dates": invalid_date_count,
            "Status": (
                "Parsed"
                if valid_date_count > 0
                else "Could not parse - original preserved"
            )
        })

    print("Date columns identified:", len(date_columns))

    # --------------------------------------------------------
    # 7.4 DETECT NUMERIC-LOOKING TEXT COLUMNS
    # --------------------------------------------------------

    for column in df.select_dtypes(
        include=["object", "string"]
    ).columns:

        non_null_values = df[column].dropna()

        if len(non_null_values) == 0:
            continue

        numeric_values = pd.to_numeric(
            non_null_values,
            errors="coerce"
        )

        numeric_percentage = (
            numeric_values.notna().mean() * 100
        )

        # Report potential numeric columns.
        # Do not convert automatically because IDs,
        # postal codes and other identifiers may be numeric-looking.
        if numeric_percentage >= 80:

            numeric_string_reports.append({
                "Dataset": dataset_name,
                "Column": column,
                "Numeric_Convertible_Percentage": round(
                    numeric_percentage, 2
                ),
                "Action": "Flagged for manual review; not converted"
            })

    # --------------------------------------------------------
    # 7.5 IDENTIFY EXACT DUPLICATE ROWS
    # --------------------------------------------------------

    exact_duplicate_count = int(df.duplicated().sum())

    # Important:
    # Exact duplicate records are reported but retained.
    # We will decide whether removal is appropriate after
    # examining the records and their business meaning.

    print("Exact duplicate rows detected:", exact_duplicate_count)

    duplicate_reports.append({
        "Dataset": dataset_name,
        "Total_Rows": len(df),
        "Exact_Duplicate_Rows": exact_duplicate_count,
        "Action": "Retained for review"
    })

    # --------------------------------------------------------
    # 7.6 PRIMARY KEY VALIDATION
    # --------------------------------------------------------

    primary_key = PRIMARY_KEYS.get(dataset_name)

    if primary_key:

        if primary_key in df.columns:

            null_key_count = int(
                df[primary_key].isna().sum()
            )

            duplicate_key_count = int(
                df[primary_key].duplicated().sum()
            )

            primary_key_status = (
                "PASS"
                if null_key_count == 0
                and duplicate_key_count == 0
                else "CHECK REQUIRED"
            )

        else:

            null_key_count = None
            duplicate_key_count = None
            primary_key_status = "PRIMARY KEY COLUMN NOT FOUND"

        primary_key_reports.append({
            "Dataset": dataset_name,
            "Primary_Key": primary_key,
            "Null_Keys": null_key_count,
            "Duplicate_Keys": duplicate_key_count,
            "Status": primary_key_status
        })

    # --------------------------------------------------------
    # 7.7 MISSING VALUE ANALYSIS AFTER CLEANING
    # --------------------------------------------------------

    for column in df.columns:

        missing_count = int(df[column].isna().sum())

        missing_percentage = (
            missing_count / len(df) * 100
            if len(df) > 0 else 0
        )

        missing_value_reports.append({
            "Dataset": dataset_name,
            "Column": column,
            "Missing_Count": missing_count,
            "Missing_Percentage": round(
                missing_percentage, 2
            )
        })

    # --------------------------------------------------------
    # 7.8 RECORD COLUMN-LEVEL CHANGES
    # --------------------------------------------------------

    for column in df.columns:

        column_reports.append({
            "Dataset": dataset_name,
            "Column": column,
            "Data_Type_After_Cleaning": str(df[column].dtype),
            "Non_Null_Count": int(df[column].notna().sum()),
            "Missing_Count": int(df[column].isna().sum()),
            "Unique_Values": int(df[column].nunique(dropna=True))
        })

    # --------------------------------------------------------
    # 7.9 STORE CLEANED DATASET
    # --------------------------------------------------------

    cleaned_datasets[dataset_name] = df.copy()

    cleaned_rows = len(df)

    cleaning_summary.append({
        "Dataset": dataset_name,
        "Original_Rows": original_rows,
        "Cleaned_Rows": cleaned_rows,
        "Original_Columns": original_columns,
        "Cleaned_Columns": len(df.columns),
        "Exact_Duplicates_Detected": exact_duplicate_count,
        "Rows_Removed": original_rows - cleaned_rows,
        "Date_Columns_Identified": len(date_columns),
        "Status": "Completed"
    })

    print("Original shape:", original_df.shape)
    print("Cleaned shape:", df.shape)
    print("Rows removed:", original_rows - cleaned_rows)
    print("Cleaning completed.")


# ------------------------------------------------------------
# 8. CREATE CLEANING REPORT DATAFRAMES
# ------------------------------------------------------------

cleaning_summary_df = pd.DataFrame(cleaning_summary)

missing_report_df = pd.DataFrame(
    missing_value_reports,
    columns=[
        "Dataset",
        "Column",
        "Missing_Count",
        "Missing_Percentage"
    ]
)

duplicate_report_df = pd.DataFrame(
    duplicate_reports,
    columns=[
        "Dataset",
        "Total_Rows",
        "Exact_Duplicate_Rows",
        "Action"
    ]
)

primary_key_report_df = pd.DataFrame(
    primary_key_reports,
    columns=[
        "Dataset",
        "Primary_Key",
        "Null_Keys",
        "Duplicate_Keys",
        "Status"
    ]
)

date_report_df = pd.DataFrame(
    date_reports,
    columns=[
        "Dataset",
        "Date_Column",
        "Valid_Dates",
        "Invalid_Dates",
        "Status"
    ]
)

column_report_df = pd.DataFrame(
    column_reports,
    columns=[
        "Dataset",
        "Column",
        "Data_Type_After_Cleaning",
        "Non_Null_Count",
        "Missing_Count",
        "Unique_Values"
    ]
)

numeric_string_report_df = pd.DataFrame(
    numeric_string_reports,
    columns=[
        "Dataset",
        "Column",
        "Numeric_Convertible_Percentage",
        "Action"
    ]
)


# ------------------------------------------------------------
# 9. DISPLAY CLEANING SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATA CLEANING SUMMARY")
print("=" * 80)

display(cleaning_summary_df)

print("\nMISSING VALUE REPORT")
display(
    missing_report_df.sort_values(
        by="Missing_Percentage",
        ascending=False
    )
)

print("\nDUPLICATE RECORD REPORT")
display(duplicate_report_df)

print("\nPRIMARY KEY REPORT")
display(primary_key_report_df)

print("\nDATE CONVERSION REPORT")
display(date_report_df)

print("\nNUMERIC-LOOKING TEXT COLUMNS")
display(numeric_string_report_df)


# ------------------------------------------------------------
# 10. SAVE CLEANED CSV DATASETS
# ------------------------------------------------------------

print("\nSAVING CLEANED DATASETS")
print("-" * 60)

for dataset_name, df in cleaned_datasets.items():

    output_file = (
        PROCESSED_DATA_PATH
        / f"{dataset_name}_cleaned.csv"
    )

    df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"{dataset_name:25} | "
        f"Saved | "
        f"{output_file.name}"
    )

print("\nAll cleaned datasets saved successfully.")


# ------------------------------------------------------------
# 11. SAVE CLEANING REPORTS TO EXCEL
# ------------------------------------------------------------

CLEANING_REPORT_FILE = (
    REPORT_PATH / "data_cleaning_report.xlsx"
)

with pd.ExcelWriter(
    CLEANING_REPORT_FILE,
    engine="openpyxl"
) as writer:

    cleaning_summary_df.to_excel(
        writer,
        sheet_name="Cleaning_Summary",
        index=False
    )

    missing_report_df.to_excel(
        writer,
        sheet_name="Missing_Values",
        index=False
    )

    duplicate_report_df.to_excel(
        writer,
        sheet_name="Duplicate_Records",
        index=False
    )

    primary_key_report_df.to_excel(
        writer,
        sheet_name="Primary_Key_Audit",
        index=False
    )

    date_report_df.to_excel(
        writer,
        sheet_name="Date_Conversion",
        index=False
    )

    column_report_df.to_excel(
        writer,
        sheet_name="Column_Summary",
        index=False
    )

    numeric_string_report_df.to_excel(
        writer,
        sheet_name="Numeric_Text_Review",
        index=False
    )

print("\nCleaning report saved:")
print(CLEANING_REPORT_FILE)


# ------------------------------------------------------------
# 12. FINAL VALIDATION
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("ZEROQ AI - DATA CLEANING COMPLETED")
print("=" * 70)

print("Total raw datasets loaded:", len(raw_datasets))
print("Total cleaned datasets:", len(cleaned_datasets))

print(
    "Total original records:",
    sum(len(df) for df in raw_datasets.values())
)

print(
    "Total cleaned records:",
    sum(len(df) for df in cleaned_datasets.values())
)

print(
    "Total rows removed:",
    sum(len(raw_datasets[name]) - len(cleaned_datasets[name])
        for name in cleaned_datasets)
)

print("\nProcessed data folder:")
print(PROCESSED_DATA_PATH)

print("\nCleaning report:")
print(CLEANING_REPORT_FILE)

print("=" * 70)
print("NOTE: Missing values and duplicate records were preserved.")
print("Original raw CSV files were not modified.")
print("=" * 70)


Libraries imported successfully.

Searching for the raw data folder...

PROJECT PATHS
------------------------------------------------------------
Project: d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training
Raw data: d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\raw_data
Processed data: d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\processed_data
Reports: d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\preprocessing\reports
products                  | Loaded successfully | Shape: (20, 6)
customers                 | Loaded successfully | Shape: (500, 5)
facilities                | Loaded successfully | Shape: (5, 4)
production_lines          | Loaded successfully | Shape: (15, 6)
materials                 | Loaded successfully | Shape: (40, 6)
production_batches        | Loaded successfully | Shape: (24000, 20)
complaints                | Loaded successfully | Shape: (6000, 14)
quality_inspections       | Loaded successfully | Shape: (37245, 11)
investig

,Dataset,Original_Rows,Cleaned_Rows,Original_Columns,Cleaned_Columns,Exact_Duplicates_Detected,Rows_Removed,Date_Columns_Identified,Status
0,products,20,20,6,6,0,0,0,Completed
1,customers,500,500,5,5,0,0,0,Completed
2,facilities,5,5,4,4,0,0,0,Completed
3,production_lines,15,15,6,6,0,0,0,Completed
4,materials,40,40,6,6,0,0,0,Completed
5,production_batches,24000,24000,20,20,0,0,2,Completed
6,complaints,6000,6000,14,14,0,0,1,Completed
7,quality_inspections,37245,37245,11,11,0,0,1,Completed
8,investigations,4000,4000,10,10,0,0,1,Completed
9,corrective_actions,3500,3500,13,13,0,0,2,Completed



MISSING VALUE REPORT


,Dataset,Column,Missing_Count,Missing_Percentage
0,products,product_id,0,0.0
1,products,product_name,0,0.0
2,products,product_category,0,0.0
3,products,storage_condition,0,0.0
4,products,shelf_life_days,0,0.0
5,products,standard_unit,0,0.0
6,customers,customer_id,0,0.0
7,customers,customer_type,0,0.0
8,customers,region,0,0.0
9,customers,customer_tenure_years,0,0.0



DUPLICATE RECORD REPORT


,Dataset,Total_Rows,Exact_Duplicate_Rows,Action
0,products,20,0,Retained for review
1,customers,500,0,Retained for review
2,facilities,5,0,Retained for review
3,production_lines,15,0,Retained for review
4,materials,40,0,Retained for review
5,production_batches,24000,0,Retained for review
6,complaints,6000,0,Retained for review
7,quality_inspections,37245,0,Retained for review
8,investigations,4000,0,Retained for review
9,corrective_actions,3500,0,Retained for review


... (3 outputs left out)

,Dataset,Date_Column,Valid_Dates,Invalid_Dates,Status
0,production_batches,production_date,24000,0,Parsed
1,production_batches,planned_dispatch_date,24000,0,Parsed
2,complaints,complaint_date,6000,0,Parsed
3,quality_inspections,inspection_date,37245,0,Parsed
4,investigations,investigation_date,4000,0,Parsed
5,corrective_actions,action_implementation_date,3500,0,Parsed
6,corrective_actions,effectiveness_review_date,3500,0,Parsed



NUMERIC-LOOKING TEXT COLUMNS


,Dataset,Column,Numeric_Convertible_Percentage,Action



SAVING CLEANED DATASETS
------------------------------------------------------------
products                  | Saved | products_cleaned.csv
customers                 | Saved | customers_cleaned.csv
facilities                | Saved | facilities_cleaned.csv
production_lines          | Saved | production_lines_cleaned.csv
materials                 | Saved | materials_cleaned.csv
production_batches        | Saved | production_batches_cleaned.csv
complaints                | Saved | complaints_cleaned.csv
quality_inspections       | Saved | quality_inspections_cleaned.csv
investigations            | Saved | investigations_cleaned.csv
corrective_actions        | Saved | corrective_actions_cleaned.csv
data_dictionary           | Saved | data_dictionary_cleaned.csv

All cleaned datasets saved successfully.

Cleaning report saved:
d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\preprocessing\reports\data_cleaning_report.xlsx


ZEROQ AI - DATA CLEANING COMPLETED
Total raw datasets loa